# IGP: QLoRA Fine-tuning — Llama 3.2 3B on Kaggle T4
**Karan — UWE Bristol MSc Data Science IGP**

### Before running:
1. Enable GPU: Settings > Accelerator > GPU T4 x2
2. Add your HuggingFace token: Settings > Secrets > Add new secret: name=`HF_TOKEN`, value=your token
   - Get token at: https://huggingface.co/settings/tokens
   - Accept Llama 3 licence at: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
3. Upload sme_train.jsonl and sme_val.jsonl as a Kaggle dataset, then add it as input

In [ ]:
import os, torch

# Load HF token from Kaggle secret
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
# Step 2: Install packages
# This takes 2-3 minutes on first run
!pip install -q \
  "transformers==4.45.2" \
  "trl==0.11.4" \
  "peft>=0.11.0" \
  "bitsandbytes==0.45.5" \
  "accelerate>=0.26.0" \
  datasets

In [ ]:
# Step 3: Upload your data files using the Kaggle sidebar
TRAIN_FILE = '/kaggle/input/datasets/karanhomayounfar1/sme-voice-data/sme_train.jsonl'
VAL_FILE   = '/kaggle/input/datasets/karanhomayounfar1/sme-voice-data/sme_val.jsonl'
OUTPUT_DIR = '/kaggle/working/checkpoints/sme-llama3-qlora'
MODEL_ID   = 'meta-llama/Llama-3.2-3B-Instruct'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Paths set.')

In [ ]:
# Step 4: Load model + tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, padding_side='right')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

# T4 = Turing, no flash_attention_2
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, token=HF_TOKEN, quantization_config=bnb,
    device_map='auto', torch_dtype=torch.bfloat16, attn_implementation='eager',
)
model.config.use_cache = False
print('Model loaded.')

In [ ]:
# Step 5: Apply QLoRA
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', inference_mode=False,
))
model.print_trainable_parameters()

In [ ]:
# Step 6: Load and format dataset
from datasets import load_dataset

def fmt(rec):
    return {'text': (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f"{rec['instruction']}<|eot_id|>"
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f"{rec['input']}<|eot_id|>"
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
        f"{rec['output']}<|eot_id|>"
    )}

train_ds = load_dataset('json', data_files=TRAIN_FILE, split='train').map(fmt)
val_ds   = load_dataset('json', data_files=VAL_FILE,   split='train').map(fmt)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# Step 7: Train
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

collator = DataCollatorForCompletionOnlyLM(
    response_template='<|start_header_id|>assistant<|end_header_id|>\n\n',
    tokenizer=tokenizer,
)

trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        gradient_checkpointing=True,
        optim='paged_adamw_32bit',
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        logging_steps=10,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        bf16=True, tf32=False,
        max_grad_norm=0.3,
        report_to='none',
        dataset_text_field='text',
        max_seq_length=512,
    ),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)

print('Starting training... (expect ~40-50 min on T4 for Llama 3.2 3B)')
trainer.train()

In [ ]:
import json
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(f'{OUTPUT_DIR}/training_meta.json','w') as f:
    json.dump({'model_id': MODEL_ID, 'lora_r': 16, 'lora_alpha': 32,
               'epochs': 3, 'lr': 2e-4, 'chat_template': 'llama3',
               'train_samples': len(train_ds)}, f, indent=2)
print(f'Done. Download from Output tab > {OUTPUT_DIR}')